# 🧪 Curiosity On Demand - Research, Evaluator & ScriptWriter Agent Testing

This notebook allows you to test and inspect the **Researcher Agent**, **Evaluator Agent**, and **ScriptWriter Agent** using Gemini and Google Search grounding in an end-to-end refinement loop.

### Workflow:
1. **Input Topic & Target Duration:** Submit any curiosity (e.g. 5 min video or 10 min deep-dive).
2. **Researcher Agent (`ResearcherAgent`):** Unconstrained search-grounded text investigation using Google Search.
3. **Evaluator Agent (`EvaluatorAgent`):** Filters noise/duplicates, structures into `ResearchReport`, and assesses sufficiency for the target duration.
4. **Iterative Loop:** Refines missing areas up to `MAX_RESEARCH_ITERATIONS`.
5. **ScriptWriter Agent (`ScriptWriterAgent`):** Generates scene-by-scene narration scripts with spoken voiceovers and summaries.

## 1. Setup & Environment Configuration

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

# Ensure project root is in sys.path when running from notebooks/
project_root = (
    Path.cwd().resolve().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd().resolve()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load environment variables (.env)
load_dotenv(project_root / ".env")

from agents.evaluator import EvaluatorAgent
from agents.researcher import ResearcherAgent
from agents.script_writer import ScriptWriterAgent
from config.settings import get_settings
from core.models import EvaluationResult, ResearchReport, Script

settings = get_settings()
print("✅ Environment initialized successfully.")
print(f"• Gemini Model:     {settings.gemini_model}")
print(f"• Research Model:   {settings.research_model}")
print(f"• Evaluator Model:  {settings.evaluator_model}")
print(f"• Script Model:     {settings.script_model}")
print(f"• Max Iterations:   {settings.max_research_iterations}")
print(
    f"• API Key Set:      {'Yes' if settings.gemini_api_key else 'No (using default credentials)'}"
)

## 2. Define Topic & Target Duration
Set the topic and optional duration you want the agent to research.

In [ ]:
TOPIC = "The Antikythera Mechanism: The Ancient Greek Analog Computer"
TARGET_DURATION = "~10 minutes"
print(f"🎯 Selected Topic: '{TOPIC}' (Target Duration: {TARGET_DURATION})")

## 3. Run Iterative Research & Evaluation Loop
Execute the researcher and evaluator multi-turn feedback loop.

In [ ]:
researcher = ResearcherAgent(settings=settings)
evaluator = EvaluatorAgent(settings=settings)

accumulated_notes = ""
feedback = None
final_report: ResearchReport | None = None
max_iterations = settings.max_research_iterations

for iteration in range(1, max_iterations + 1):
    print(f"\n🔄 --- Iteration Round {iteration}/{max_iterations} ---")
    print("🔍 Researcher gathering facts...")
    round_notes = await researcher.conduct_research(
        topic=TOPIC,
        feedback=feedback,
        previous_research=accumulated_notes if accumulated_notes else None,
        target_duration=TARGET_DURATION,
    )

    accumulated_notes = (
        f"{accumulated_notes}\n\n=== ROUND {iteration} ===\n{round_notes}"
        if accumulated_notes
        else round_notes
    )

    print("⚖️ Evaluator assessing quality and structuring schema...")
    eval_result: EvaluationResult = await evaluator.evaluate_research(
        topic=TOPIC,
        accumulated_research=accumulated_notes,
        target_duration=TARGET_DURATION,
        iteration=iteration,
        max_iterations=max_iterations,
    )

    final_report = eval_result.report

    if eval_result.is_sufficient or iteration >= max_iterations:
        print(f"✅ Research finalized! Sufficient: {eval_result.is_sufficient}")
        break

    feedback = eval_result.feedback_for_researcher
    print(f"⚠️ Content incomplete. Evaluator feedback for next round:\n{feedback}")

## 4. Inspect Researched Information
Display the high-level summary and all researched subtopics with their contents.

In [ ]:
if final_report:
    print(f"📋 TOPIC: {final_report.topic}\n")
    print(f"💡 SUMMARY:\n{final_report.summary}\n")
    print("" + "=" * 60 + "\n")
    print(f"📚 RESEARCHED SUBTOPICS ({len(final_report.sections)} sections):\n")

    for i, section in enumerate(final_report.sections, 1):
        print(f"[{i}] 📌 {section.topic}")
        print(f"{section.content}")
        print("-" * 60)

## 5. Generate Scene-by-Scene Narration Script
Use `ScriptWriterAgent` to convert the structured research report into a narration script.

In [ ]:
if final_report is not None:
    script_writer = ScriptWriterAgent(settings=settings)
    print("✍️ Generating scene-by-scene narration script...")

    generated_script: Script = await script_writer.generate_script(
        topic=TOPIC,
        research_report=final_report,
        target_duration=TARGET_DURATION,
    )

    print(f"✅ Script Generated: '{generated_script.topic}'")
    print(f"📝 Overview: {generated_script.summary}\n")
    print("=" * 60 + "\n")

    for scene in generated_script.scenes:
        print(f"🎬 Scene {scene.scene_number}: {scene.title}")
        print(f"📌 Summary: {scene.summary}")
        print(f"🎙️ Narration: {scene.narration}")
        print("-" * 60)
else:
    print("⚠️ No final_report available. Please run the research loop first.")